In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
import os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
class SimpleBoundingBoxRegressor(nn.Module):
    """
    简化的边界框回归模型
    输入: (batch_size, 64, 64, 256) 或 (batch_size, 256, 64, 64) 的特征图
    输出: 每个图片一个边界框 (x, y, w, h) 真实像素坐标
    """
    
    def __init__(self, input_channels=256, spatial_size=64, img_width=512, img_height=512):
        super(SimpleBoundingBoxRegressor, self).__init__()
        
        self.input_channels = input_channels
        self.spatial_size = spatial_size
        self.img_width = img_width
        self.img_height = img_height
        
        # 特征处理层 - 只需要一层卷积
        # 输入: (batch_size, 256, 64, 64)
        # 输出: (batch_size, 512, 32, 32)
        self.feature_processor = nn.Sequential(
            nn.Conv2d(input_channels, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 64x64 -> 32x32
        )
        
        # 全局平均池化后的特征维度
        pooled_size = 32
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # 回归头
        # 输入: 512维特征向量
        # 输出: 4个值 (x, y, w, h) 真实像素坐标
        self.regressor = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, 4)  # 输出: x, y, w, h (真实像素坐标)
        )
        
        # 初始化权重
        self._initialize_weights()
    
    def _initialize_weights(self):
        """初始化权重"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        """
        前向传播
        
        参数:
            x: 输入特征图, shape: (batch_size, 256, 64, 64) 或 (batch_size, 64, 64, 256)
        
        返回:
            bbox: 边界框, shape: (batch_size, 4)
                  格式: [x_center, y_center, width, height] (真实像素坐标)
        """
        # 确保输入形状正确
        if x.dim() == 4:
            # 如果是 (batch, H, W, C) 转换为 (batch, C, H, W)
            if x.shape[-1] == 256:  # NHWC格式
                x = x.permute(0, 3, 1, 2)  # 转换为NCHW
        else:
            raise ValueError(f"输入维度错误: {x.shape}")
        
        # 特征处理
        features = self.feature_processor(x)  # (batch, 512, 32, 32)
        
        # 全局池化
        features = self.global_pool(features)  # (batch, 512, 1, 1)
        
        # 展平
        features = features.view(features.size(0), -1)  # (batch, 512)
        
        # 回归
        bbox = self.regressor(features)  # (batch, 4)
        
        # 确保输出为正数（真实像素坐标）
        # 使用ReLU确保非负，但也可以使用softplus
        bbox = F.relu(bbox)
        
        return bbox
    
    def convert_to_pixel_coords(self, bbox):
        """
        将模型输出转换为像素坐标
        
        参数:
            bbox: 模型输出，格式可能是各种归一化或缩放
        """
        # 这里假设模型直接输出像素坐标
        # 如果需要调整，可以在这里进行缩放
        return bbox

class PixelBBoxLoss(nn.Module):
    """
    真实像素坐标的边界框损失函数
    """
    
    def __init__(self, img_width=512, img_height=512, 
                 lambda_l1=1.0, lambda_iou=5.0, lambda_size=0.1,
                 reduction='mean'):
        super(PixelBBoxLoss, self).__init__()
        
        self.img_width = img_width
        self.img_height = img_height
        self.lambda_l1 = lambda_l1
        self.lambda_iou = lambda_iou
        self.lambda_size = lambda_size
        self.reduction = reduction
        
    def forward(self, pred, target):
        """
        计算损失
        
        参数:
            pred: 预测边界框 [x, y, w, h] (像素坐标)
            target: 真实边界框 [x, y, w, h] (像素坐标)
        """
        # 1. L1损失
        l1_loss = F.l1_loss(pred, target, reduction=self.reduction)
        
        # 2. IoU损失
        iou_loss = self.iou_loss(pred, target)
        
        # 3. 尺寸合理性损失
        size_loss = self.size_consistency_loss(pred)
        
        # 4. 中心点位置损失（额外惩罚中心点偏移）
        center_loss = F.l1_loss(pred[:, :2], target[:, :2], reduction=self.reduction)
        
        # 总损失
        total_loss = (
            self.lambda_l1 * l1_loss +
            self.lambda_iou * iou_loss +
            self.lambda_size * size_loss +
            0.5 * center_loss
        )
        
        return {
            'total_loss': total_loss,
            'l1_loss': l1_loss,
            'iou_loss': iou_loss,
            'size_loss': size_loss,
            'center_loss': center_loss
        }
    
    def iou_loss(self, pred, target):
        """计算IoU损失 (1 - IoU)"""
        # 将像素坐标转换为角点坐标
        pred_corners = self.pixel_to_corners(pred)
        target_corners = self.pixel_to_corners(target)
        
        # 计算交集
        inter_x1 = torch.max(pred_corners[:, 0], target_corners[:, 0])
        inter_y1 = torch.max(pred_corners[:, 1], target_corners[:, 1])
        inter_x2 = torch.min(pred_corners[:, 2], target_corners[:, 2])
        inter_y2 = torch.min(pred_corners[:, 3], target_corners[:, 3])
        
        inter_area = torch.clamp(inter_x2 - inter_x1, min=0) * torch.clamp(inter_y2 - inter_y1, min=0)
        
        # 计算并集
        pred_area = (pred_corners[:, 2] - pred_corners[:, 0]) * (pred_corners[:, 3] - pred_corners[:, 1])
        target_area = (target_corners[:, 2] - target_corners[:, 0]) * (target_corners[:, 3] - target_corners[:, 1])
        union_area = pred_area + target_area - inter_area + 1e-6
        
        # IoU
        iou = inter_area / union_area
        
        # IoU损失
        iou_loss = 1.0 - iou
        
        if self.reduction == 'mean':
            return iou_loss.mean()
        elif self.reduction == 'sum':
            return iou_loss.sum()
        else:
            return iou_loss
    
    def pixel_to_corners(self, boxes):
        """将[x_center, y_center, width, height]像素坐标转换为[x1, y1, x2, y2]"""
        x_center, y_center, width, height = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
        
        x1 = x_center - width / 2
        y1 = y_center - height / 2
        x2 = x_center + width / 2
        y2 = y_center + height / 2
        
        return torch.stack([x1, y1, x2, y2], dim=1)
    
    def size_consistency_loss(self, pred):
        """尺寸合理性损失"""
        width, height = pred[:, 2], pred[:, 3]
        
        # 鼓励合理的尺寸（不超出图像范围，不过小）
        # 最小尺寸约束
        min_size = 20.0
        width_loss = torch.relu(min_size - width)
        height_loss = torch.relu(min_size - height)
        
        # 最大尺寸约束（不超过图像尺寸）
        max_width = self.img_width
        max_height = self.img_height
        width_max_loss = torch.relu(width - max_width)
        height_max_loss = torch.relu(height - max_height)
        
        size_loss = (width_loss + height_loss + width_max_loss + height_max_loss).mean()
        
        return size_loss


In [30]:
class RealCoordDataset(Dataset):

    def __init__(self, files, boxes, folder = "../SAM_b_features/image_feature"):

        self.folder = folder
        self.files = files
        self.boxes = []
        for b in boxes:
            with open(box_sort[0], "r") as f:
                cor = list(map(int, f.readline().split(" ")))
            self.boxes.append(cor)
                
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        features = torch.load(os.path.join(self.folder, self.files[idx]))
        bbox = self.boxes[idx]
         
        return torch.FloatTensor(features), torch.FloatTensor(bbox)

In [31]:
def file_generator(folder_path = "../datasets"):
    """
    file path generator
    """
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            yield os.path.join(root, file)

datafolder = "../SAM_b_features/image_feature"
files = os.listdir(datafolder)
file_sort = sorted(files)

boxes = [x for x in file_generator("../datasets") if ".txt" in x]
box_sort = sorted(boxes)

X_train, X_val, y_train, y_val = train_test_split(
    file_sort, box_sort,
    test_size=0.2,
    random_state=42
)

train_data = RealCoordDataset(X_train, y_train)
val_data = RealCoordDataset(X_val, y_val)